# **WEEK - 3 ASSIGNMENT**

Objective: 

Using Subqueries, CTEs, and Window Functions to analyze sales data from the Superstore dataset.

> **Sample Data from superstore_raw**

In [0]:
%sql
select * from superstore_raw limit 10;

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


### **Creation of Tables**
- **Customers Table**
- **Orders Table**
- **Products Table**

In [0]:
%sql
create table customers (
customer_id string,
customer_name string,
segment string,
country string,
city string,
state string,
postal_code int,
region string
);

In [0]:
%sql
create table orders (
id int,
order_id string,
order_date date,
ship_date date,
ship_mode string,
customer_id string,
product_id string,
sales double,
quantity int,
discount double,
profit double
);

In [0]:
%sql
create table products (
product_id string,
product_name string,
category string,
sub_category string
);

### **Inserting Data in Tables**

In [0]:
%sql
insert into customers
select distinct
`Customer ID`,
`Customer Name`,
Segment,
Country,
City,
State,
`Postal Code`,
Region
from superstore_raw;

num_affected_rows,num_inserted_rows
4910,4910


In [0]:
%sql
insert into orders
select distinct
`Row ID`,
`Order ID`,
`Order Date`,
`Ship Date`,
`Ship Mode`,
`Customer ID`,
`Product ID`,
Sales,
Quantity,
Discount,
Profit
from superstore_raw;

num_affected_rows,num_inserted_rows
9994,9994


In [0]:
%sql
insert into products
select distinct `Product ID`, 
`Product Name`,
Category,
`Sub-Category`
from superstore_raw;

num_affected_rows,num_inserted_rows
1894,1894


### Step 2: Perform Required Queries

**Note**: **I used `SELECT DISTINCT customer_id, customer_name FROM customers` everywhere for joining instead of directly joining with the `customers` table, because the dataset has multiple rows for the same customer with different addresses, which was creating duplicate results after joining. So, this way i  used a distinct customer list to make sure each customer is joined only once.**


In [0]:
%sql
-- Q1 : find all orders where sales are greater than average sales
select *, (select round(avg(sales), 2) from orders) as avg_sales
from orders
where sales > (select avg(sales) from orders);

id,order_id,order_date,ship_date,ship_mode,customer_id,product_id,sales,quantity,discount,profit,avg_sales
86,CA-2017-140088,2017-05-28,2017-05-30,Second Class,PO-18865,FUR-CH-10000863,301.96,2,0.0,33.2156,229.86
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,FUR-TA-10001768,787.53,3,0.0,165.3813,229.86
150,CA-2016-114489,2016-12-05,2016-12-09,Standard Class,JE-16165,FUR-CH-10000454,1951.84,8,0.0,585.552,229.86
219,CA-2016-130162,2016-10-28,2016-11-01,Standard Class,JH-15910,TEC-PH-10002563,302.376,3,0.2,22.6782,229.86
224,CA-2015-169397,2015-12-24,2015-12-27,First Class,JB-15925,TEC-MA-10001148,479.988,4,0.7,-383.9904,229.86
236,US-2017-100930,2017-04-07,2017-04-12,Standard Class,CS-12400,TEC-AC-10003832,617.976,3,0.2,-7.7247,229.86
254,CA-2016-146941,2016-12-10,2016-12-13,First Class,DL-13315,OFF-EN-10003296,361.92,4,0.0,162.864,229.86
339,CA-2014-129924,2014-07-12,2014-07-17,Standard Class,AC-10420,FUR-TA-10004575,698.352,3,0.2,-17.4588,229.86
355,CA-2016-138520,2016-04-08,2016-04-13,Standard Class,JL-15505,FUR-BO-10002268,388.704,6,0.2,-4.8588,229.86
376,US-2014-119137,2014-07-23,2014-07-27,Standard Class,AG-10900,TEC-AC-10002076,479.04,10,0.2,-29.94,229.86


In [0]:
%sql
-- Q2: find the highest sales order for each customer.
select o.customer_id, c.customer_name, o.order_id, o.sales as highest_sale
from orders o
join (select distinct customer_id, customer_name from customers) c 
on o.customer_id = c.customer_id
where o.sales = (
    select max(sales)
    from orders o2 where o2.customer_id = o.customer_id
)
order by o.customer_id;

customer_id,customer_name,order_id,highest_sale
AA-10315,Alex Avila,CA-2016-103982,3930.072
AA-10375,Allen Armold,CA-2016-131065,499.98
AA-10480,Andrew Allen,CA-2016-114601,479.97
AA-10645,Anna Andreadi,CA-2015-110863,1323.9
AB-10015,Aaron Bergman,CA-2016-140935,341.96
AB-10060,Adam Bellavance,CA-2016-129714,4355.168
AB-10105,Adrian Barton,CA-2016-117121,9892.74
AB-10150,Aimee Bixby,CA-2014-169061,479.97
AB-10165,Alan Barnes,CA-2015-115945,304.23
AB-10255,Alejandro Ballentine,CA-2017-126662,479.984


In [0]:
%sql
-- Q3: calculate total sales for each customer
with customer_sales as (
    select customer_id, round(sum(sales), 2) as total_sales
    from orders
    group by customer_id
)
select c.customer_id, c.customer_name, cs.total_sales
from customer_sales cs
join (select distinct customer_id, customer_name from customers) c
on cs.customer_id = c.customer_id;


customer_id,customer_name,total_sales
JK-15640,Jim Kriz,4760.43
BK-11260,Berenike Kampe,659.14
TC-21475,Tony Chapman,1244.98
JL-15835,John Lee,9799.92
AA-10645,Anna Andreadi,5086.93
RD-19720,Roger Demir,1419.74
JB-15925,Joni Blumstein,900.55
SS-20410,Shahid Shariari,3056.81
JS-15940,Joni Sundaresam,469.17
KD-16495,Keith Dawkins,8181.26


In [0]:
%sql
-- Q4: find customers whose total sales are above average
with customer_sales as (
    select customer_id, round(sum(sales),2) as total_sales
    from orders
    group by customer_id
)
select c.customer_id, c.customer_name, cs.total_sales, (select round(avg(sales), 2) from orders) as avg_sales
from customer_sales cs
join (select distinct customer_id, customer_name from customers) c 
on cs.customer_id = c.customer_id
where cs.total_sales > (select avg(sales) from orders)
order by c.customer_id;

customer_id,customer_name,total_sales,avg_sales
AA-10315,Alex Avila,5563.56,229.86
AA-10375,Allen Armold,1056.39,229.86
AA-10480,Andrew Allen,1790.51,229.86
AA-10645,Anna Andreadi,5086.93,229.86
AB-10015,Aaron Bergman,886.16,229.86
AB-10060,Adam Bellavance,7755.62,229.86
AB-10105,Adrian Barton,14473.57,229.86
AB-10150,Aimee Bixby,966.71,229.86
AB-10165,Alan Barnes,1113.84,229.86
AB-10255,Alejandro Ballentine,914.53,229.86


In [0]:
%sql
--Q5: rank all customers based on total sales
select c.customer_id, c.customer_name, round(sum(o.sales),2) as total_sales,
rank() over (order by sum(o.sales) desc) as sales_rank
from orders o
join (select distinct customer_id, customer_name from customers) as c
on o.customer_id = c.customer_id
group by c.customer_id, c.customer_name;

customer_id,customer_name,total_sales,sales_rank
SM-20320,Sean Miller,25043.05,1
TC-20980,Tamara Chand,19052.22,2
RB-19360,Raymond Buch,15117.34,3
TA-21385,Tom Ashbrook,14595.62,4
AB-10105,Adrian Barton,14473.57,5
KL-16645,Ken Lonsdale,14175.23,6
SC-20095,Sanjit Chand,14142.33,7
HL-15040,Hunter Lopez,12873.3,8
SE-20110,Sanjit Engle,12209.44,9
CC-12370,Christopher Conant,12129.07,10


In [0]:
%sql
-- Q6: assign row numbers to each order within a customer
select o.customer_id, c.customer_name, o.order_id, o.sales,
row_number() over (partition by o.customer_id order by o.sales desc) as order_rank
from orders o
join (select distinct customer_id, customer_name from customers) c 
on o.customer_id = c.customer_id;

customer_id,customer_name,order_id,sales,order_rank
AA-10315,Alex Avila,CA-2016-103982,3930.072,1
AA-10315,Alex Avila,CA-2014-128055,673.568,2
AA-10315,Alex Avila,CA-2016-103982,431.976,3
AA-10315,Alex Avila,CA-2017-147039,362.94,4
AA-10315,Alex Avila,CA-2014-128055,52.98,5
AA-10315,Alex Avila,CA-2016-103982,41.72,6
AA-10315,Alex Avila,CA-2015-121391,26.96,7
AA-10315,Alex Avila,CA-2014-138100,14.94,8
AA-10315,Alex Avila,CA-2014-138100,14.56,9
AA-10315,Alex Avila,CA-2017-147039,11.54,10


In [0]:
%sql
--Q7 : display top 3 customers based on total sales
with customers_rank as (
    select o.customer_id, sum(o.sales) as total_sales,
    rank() over (order by sum(o.sales) desc) as sales_rank
    from orders o
    group by customer_id
)
select cr.customer_id, c.customer_name, round(cr.total_sales, 2) as total_sales, cr.sales_rank
from customers_rank cr
join (select distinct customer_id, customer_name from customers) c
on cr.customer_id = c.customer_id
where sales_rank <= 3;

customer_id,customer_name,total_sales,sales_rank
SM-20320,Sean Miller,25043.05,1
TC-20980,Tamara Chand,19052.22,2
RB-19360,Raymond Buch,15117.34,3


**Final Combined Query** (using JOIN + CTE + Window Function together)

In [0]:
%sql
with total as (
    select o.customer_id, sum(o.sales) as total_sales
    from orders o
    group by o.customer_id
)
select c.customer_name, t.total_sales, rank() over (order by t.total_sales desc) as sales_rank
from total t
join (select distinct customer_id, customer_name from customers) c
on t.customer_id = c.customer_id
order by sales_rank;

## **MINI PROJECT : Customer Sales Insights**

> Top 5 Customers

In [0]:
%sql
select c.customer_id, c.customer_name, round(sum(o.sales),2) as total_sales
from orders o
join (select distinct customer_id, customer_name from customers) c 
on o.customer_id = c.customer_id
group by c.customer_id, c.customer_name
order by total_sales desc
limit 5;

customer_id,customer_name,total_sales
SM-20320,Sean Miller,25043.05
TC-20980,Tamara Chand,19052.22
RB-19360,Raymond Buch,15117.34
TA-21385,Tom Ashbrook,14595.62
AB-10105,Adrian Barton,14473.57


> Bottom 5 customers

In [0]:
%sql
select c.customer_id, c.customer_name, round(sum(o.sales),2) as total_sales
from orders o
join (select distinct customer_id, customer_name from customers) c 
on o.customer_id = c.customer_id
group by c.customer_id, c.customer_name
order by total_sales asc
limit 5;

customer_id,customer_name,total_sales
TS-21085,Thais Sissman,4.83
LD-16855,Lela Donovan,5.3
CJ-11875,Carl Jackson,16.52
MG-18205,Mitch Gastineau,16.74
RS-19870,Roy Skaria,22.33


> Customers with only 1 order

In [0]:
%sql
select c.customer_id, c.customer_name, count(o.order_id) as order_count
from orders o
join (select distinct customer_id, customer_name from customers) c 
on o.customer_id = c.customer_id
group by c.customer_id, c.customer_name
having count(o.order_id) = 1
order by c.customer_name;

customer_id,customer_name,order_count
AO-10810,Anthony O'Donnell,1
CJ-11875,Carl Jackson,1
JR-15700,Jocasta Rupert,1
LD-16855,Lela Donovan,1
RE-19405,Ricardo Emerson,1


> Customers with above-average sales

In [0]:
%sql
with customer_sales as (
    select customer_id, sum(sales) as total_sales
    from orders
    group by customer_id
)
select c.customer_id, c.customer_name, round(cs.total_sales, 2) as total_sales
from customer_sales cs
join (select distinct customer_id, customer_name from customers) c 
on cs.customer_id = c.customer_id
where cs.total_sales > (select avg(total_sales) from customer_sales)

customer_id,customer_name,total_sales
JK-15640,Jim Kriz,4760.43
JL-15835,John Lee,9799.92
AA-10645,Anna Andreadi,5086.93
SS-20410,Shahid Shariari,3056.81
KD-16495,Keith Dawkins,8181.26
SA-20830,Sue Ann Reed,4767.34
GA-14515,George Ashbrook,3919.78
AH-10690,Anna H�berlin,7888.29
LF-17185,Luke Foster,3930.51
MD-17350,Maribeth Dona,3766.38


> Highest order value per customer

In [0]:
%sql
select c.customer_id, c.customer_name, max(o.sales) as highest_order_value
from orders o
join ( select distinct customer_id, customer_name from customers) c 
on o.customer_id = c.customer_id
group by c.customer_id, c.customer_name;

customer_id,customer_name,highest_order_value
KB-16315,Karl Braun,479.984
JB-15925,Joni Blumstein,479.988
AH-10195,Alan Haines,961.48
SA-20830,Sue Ann Reed,2735.952
BK-11260,Berenike Kampe,214.95
CC-12430,Chuck Clark,518.272
JK-15205,Jamie Kunitz,1871.88
TG-21310,Toby Gnade,1016.792
PW-19240,Pierre Wener,2541.98
GA-14515,George Ashbrook,1447.65
